## **Evaluation of No Cleaning Method**

In [1]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline
from pyspark.sql.functions import trim, col

from sparknlp.pretrained import ResourceDownloader

In [2]:
spark = sparknlp.start()

26/04/18 15:17:29 WARN Utils: Your hostname, Meenas-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.0.0.150 instead (on interface en0)
26/04/18 15:17:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/meenabalamurugan/tweet-sentiment-analysis/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/meenabalamurugan/.ivy2/cache
The jars for the packages stored in: /Users/meenabalamurugan/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f90d645d-2858-4e0b-bcd4-7ed4b3f63255;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;5.5.0 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-j

26/04/18 15:17:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### Load Data

In [3]:
CLEAN_DATA_PATH = '../cleaned_data/baseline_clean_data.csv'

In [4]:
df = spark.read.option("header", True).csv(CLEAN_DATA_PATH)
df.show(5)
print("Number of rows:", df.count())

+----+-----------+------------+--------------------+
|  id|      topic|og_sentiment|        cleaned_text|
+----+-----------+------------+--------------------+
|2401|Borderlands|    Positive|im getting on bor...|
|2401|Borderlands|    Positive|I am coming to th...|
|2401|Borderlands|    Positive|im getting on bor...|
|2401|Borderlands|    Positive|im coming on bord...|
|2401|Borderlands|    Positive|im getting on bor...|
+----+-----------+------------+--------------------+
only showing top 5 rows

Number of rows: 74682


### Drop Nulls and Empty Values

In [5]:
df = df.dropna(subset=["cleaned_text"])
print("Number of rows after dropping nulls:", df.count())

Number of rows after dropping nulls: 73996


In [6]:
from pyspark.sql.functions import trim

df = df.filter(trim(col("cleaned_text")) != "")
print("Number of rows after dropping empty strings:", df.count())

Number of rows after dropping empty strings: 73824


### Drop Rows with "Irrelevant" OG Sentiment
These must be dropped as the model can only classify "positive", "neutral", or "negative". 

In [7]:
df = df.filter(col("og_sentiment") != "Irrelevant")
print("Number of rows after dropping irrelevant rows:", df.count())

Number of rows after dropping irrelevant rows: 60982


### Run Model

In [8]:
document_assembler = DocumentAssembler() \
    .setInputCol('cleaned_text') \
    .setOutputCol('document')

tokenizer = Tokenizer() \
    .setInputCols(['document']) \
    .setOutputCol('token')

classifier = XlmRoBertaForSequenceClassification.pretrained('twitter_xlm_roberta_base_sentiment')\
  .setInputCols(["document",'token'])\
  .setOutputCol("pred_sentiment")

pipeline = Pipeline(stages=[
    document_assembler, 
    tokenizer,
    classifier    
])

model = pipeline.fit(df)
result = model.transform(df)

twitter_xlm_roberta_base_sentiment download started this may take some time.
Approximate size to download 993.6 MB
[ | ]26/04/18 15:20:35 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
twitter_xlm_roberta_base_sentiment download started this may take some time.
Approximate size to download 993.6 MB
26/04/18 15:20:35 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
[ | ]Download done! Loading the resource.
[OK!]


In [9]:
result.show()

Using CPUs
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|  id|      topic|og_sentiment|        cleaned_text|            document|               token|      pred_sentiment|
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 52...|[{token, 0, 1, im...|[{category, 0, 52...|
|2401|Borderlands|    Positive|I am coming to th...|[{document, 0, 50...|[{token, 0, 0, I,...|[{category, 0, 50...|
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 49...|[{token, 0, 1, im...|[{category, 0, 49...|
|2401|Borderlands|    Positive|im coming on bord...|[{document, 0, 50...|[{token, 0, 1, im...|[{category, 0, 50...|
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 56...|[{token, 0, 1, im...|[{category, 0, 56...|
|2401|Borderlands|    Positive|im getting into b...|[{documen

In [10]:
# View the actual labels from the pred_sentiment column
result.select("og_sentiment", "cleaned_text", "pred_sentiment.result").show()

+------------+--------------------+----------+
|og_sentiment|        cleaned_text|    result|
+------------+--------------------+----------+
|    Positive|im getting on bor...|[negative]|
|    Positive|I am coming to th...|[negative]|
|    Positive|im getting on bor...|[negative]|
|    Positive|im coming on bord...|[negative]|
|    Positive|im getting on bor...| [neutral]|
|    Positive|im getting into b...|[negative]|
|    Positive|So I spent a few ...| [neutral]|
|    Positive|So I spent a coup...| [neutral]|
|    Positive|So I spent a few ...|[positive]|
|    Positive|So I spent a few ...| [neutral]|
|    Positive|2010 So I spent a...| [neutral]|
|    Positive|                was |[negative]|
|     Neutral|Rock-Hard La Varl...| [neutral]|
|     Neutral|Rock-Hard La Varl...| [neutral]|
|     Neutral|Rock-Hard La Varl...| [neutral]|
|     Neutral|Rock-Hard La Vita...| [neutral]|
|     Neutral|Live Rock - Hard ...| [neutral]|
|     Neutral|I-Hard like me, R...| [neutral]|
|    Positive

In [11]:
# Create column with the predicted labels
result = result.withColumn("pred_label",col("pred_sentiment")[0].result)
result.show()

+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+
|  id|      topic|og_sentiment|        cleaned_text|            document|               token|      pred_sentiment|pred_label|
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 52...|[{token, 0, 1, im...|[{category, 0, 52...|  negative|
|2401|Borderlands|    Positive|I am coming to th...|[{document, 0, 50...|[{token, 0, 0, I,...|[{category, 0, 50...|  negative|
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 49...|[{token, 0, 1, im...|[{category, 0, 49...|  negative|
|2401|Borderlands|    Positive|im coming on bord...|[{document, 0, 50...|[{token, 0, 1, im...|[{category, 0, 50...|  negative|
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 56...|[{token, 0, 1, im...|[{category, 0, 56

### Normalize Label Columns

In [12]:
from pyspark.sql.functions import lower

result = result.withColumn("og_label", lower(col("og_sentiment"))) \
               .withColumn("pred_label", lower(col("pred_label")))

result.show()

+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+--------+
|  id|      topic|og_sentiment|        cleaned_text|            document|               token|      pred_sentiment|pred_label|og_label|
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+--------+
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 52...|[{token, 0, 1, im...|[{category, 0, 52...|  negative|positive|
|2401|Borderlands|    Positive|I am coming to th...|[{document, 0, 50...|[{token, 0, 0, I,...|[{category, 0, 50...|  negative|positive|
|2401|Borderlands|    Positive|im getting on bor...|[{document, 0, 49...|[{token, 0, 1, im...|[{category, 0, 49...|  negative|positive|
|2401|Borderlands|    Positive|im coming on bord...|[{document, 0, 50...|[{token, 0, 1, im...|[{category, 0, 50...|  negative|positive|
|2401|Borderlands|    Positive|im getting on bor

### Calculate Accuracy

In [14]:
eval_df = result.select("og_label", "pred_label").toPandas()
eval_df

,og_label,pred_label
0,positive,negative
1,positive,negative
2,positive,negative
3,positive,negative
4,positive,neutral
...,...,...
60977,positive,negative
60978,positive,negative
60979,positive,negative
60980,positive,neutral


In [16]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(eval_df["og_label"], eval_df["pred_label"])
print("Accuracy:", accuracy)

Accuracy: 0.5084123183890328


### Calculate F1 Scores

In [17]:
from sklearn.metrics import f1_score

macro_f1 = f1_score(eval_df["og_label"], eval_df["pred_label"], average="macro")
print("Macro F1:", macro_f1)

weighted_f1 = f1_score(eval_df["og_label"], eval_df["pred_label"], average="weighted")
print("Weighted F1:", weighted_f1)

Macro F1: 0.5117266045096062
Weighted F1: 0.5131995815482512


### Classification Report

In [18]:
from sklearn.metrics import classification_report

print(classification_report(eval_df["og_label"], eval_df["pred_label"]))

              precision    recall  f1-score   support

    negative       0.66      0.42      0.52     22312
     neutral       0.38      0.65      0.48     18051
    positive       0.62      0.47      0.54     20619

    accuracy                           0.51     60982
   macro avg       0.56      0.52      0.51     60982
weighted avg       0.57      0.51      0.51     60982

